# B03 DBSCAN: Yeo–Johnson trên RFM


## Params

In [1]:
EXPERIMENT = 'B03-DB'
FEATURES = ['Recency', 'Frequency', 'Monetary']
POWER_METHOD = 'yeo-johnson'
POWER_STANDARDIZE = False
PCA_VAR = 0.90
# Giữ cùng lưới và điều kiện chọn với B01 DBSCAN.
MIN_SAMPLES_GRID = [4, 6, 8, 10, 15, 20]
EPS_GRID = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5, 0.7, 1.0, 1.5, 2.0]
MAX_NOISE = 0.05
MIN_CLUSTERS = 2

## Import thư viện và đọc dữ liệu

In [2]:
from pathlib import Path
from hashlib import sha256
import json
import joblib
import numpy as np
import pandas as pd
import sklearn
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.preprocessing import PowerTransformer, StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, davies_bouldin_score

def find(rel):
    for parent in [Path.cwd(), *Path.cwd().parents]:
        for candidate in (parent / rel, parent / 'project' / rel):
            if candidate.exists():
                return candidate.resolve()
    raise FileNotFoundError(f'Không tìm thấy trong repository: {rel}')

BASE_DIR = find('notebooks/03_clustering_eval/17-11-2026-hieu-dbscan-b03')
# Cùng đầu vào đã tiền xử lý với B01; không thay đổi file feature.
DATA = find('notebooks/02_baseline_kmeans/12-11-2026-hieu-kmeans/data/customer_features.csv')
FIG, OUT = BASE_DIR / 'figures', BASE_DIR / 'data'
FIG.mkdir(exist_ok=True)
OUT.mkdir(exist_ok=True)
df = pd.read_csv(DATA, dtype={'CustomerID': 'string'})
X = df[FEATURES].copy()
assert df['CustomerID'].notna().all() and df['CustomerID'].is_unique
assert np.isfinite(X.to_numpy(dtype=float)).all(), 'RFM có missing hoặc giá trị vô hạn.'
assert (X >= 0).all().all(), 'Tập RFM mua đầu vào phải không âm.'
assert X.nunique().gt(1).all(), 'PowerTransformer cần các biến không hằng.'
assert len(X) >= max(MIN_SAMPLES_GRID)
print('Dữ liệu đầu vào:', X.shape)
display(X.describe().round(2))

Dữ liệu đầu vào: (4327, 3)


,Recency,Frequency,Monetary
count,4327.00,4327.00,4327.00
mean,93.04,4.25,1944.95
std,99.98,7.64,8441.89
min,1.00,1.00,2.90
25%,18.00,1.00,301.52
50%,51.00,2.00,657.90
75%,143.00,5.00,1624.10
max,374.00,206.00,278748.02


**Phạm vi:** dùng RFM đã tiền xử lý của nhóm, giữ X trên thang gốc để mô tả cụm.
Thí nghiệm này không làm lại cleaning hoặc xác minh chuỗi raw → features.
Không lấy CustomerID hay các cột hành vi khác làm đầu vào DBSCAN.

## Yeo–Johnson rồi StandardScaler

Fit Yeo–Johnson trên cả R, F, M, không áp dụng log1p trước đó.
`standardize=False` để bước power transform chỉ biến đổi phân phối;
StandardScaler được fit riêng sau biến đổi. Lambda của mỗi biến phải được giữ lại.

In [3]:
power = PowerTransformer(method=POWER_METHOD, standardize=POWER_STANDARDIZE)
X_power = power.fit_transform(X)
assert np.isfinite(X_power).all()
lambda_by_feature = dict(zip(FEATURES, map(float, power.lambdas_)))
display(pd.Series(lambda_by_feature, name='Lambda đã fit'))

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_power)
display(pd.DataFrame({
    'skew_raw': X.skew(),
    'skew_yeo_johnson': pd.DataFrame(X_power, columns=FEATURES).skew(),
}).round(3))

Recency      0.125217
Frequency   -0.770386
Monetary    -0.078253
Name: Lambda đã fit, dtype: float64

,skew_raw,skew_yeo_johnson
Recency,1.246,-0.051
Frequency,11.948,0.219
Monetary,21.036,-0.015


In [4]:
pca_full = PCA(svd_solver='full').fit(X_scaled)
cum = np.cumsum(pca_full.explained_variance_ratio_)
n_pc = min(int(np.searchsorted(cum, PCA_VAR) + 1), len(FEATURES))
pca_b03 = PCA(n_components=n_pc, svd_solver='full')
Z = pca_b03.fit_transform(X_scaled)
assert np.isfinite(Z).all()
assert pca_b03.explained_variance_ratio_.sum() >= PCA_VAR - 1e-12
print('Phương sai tích lũy:', cum.round(4))
print(f'Số PC giữ lại: {n_pc}/{len(FEATURES)}; phương sai giữ {pca_b03.explained_variance_ratio_.sum():.2%}')

Phương sai tích lũy: [0.7472 0.9334 1.    ]
Số PC giữ lại: 2/3; phương sai giữ 93.34%


**PCA:** ngưỡng là 90% phương sai, không cố định 2 chiều. Số thành phần được tính lại sau Yeo–Johnson và StandardScaler.

**Đối chiếu B01:** thay phép biến đổi log1p bằng Yeo–Johnson; giữ StandardScaler, ngưỡng PCA 90%, lưới và quy tắc chọn. Không kết luận tốt hơn B01 trước khi có kết quả B03.

## Chọn eps bằng đồ thị k-distance

Với mỗi `min_samples`, tính khoảng cách từ mỗi khách đến láng giềng thứ `min_samples` (tính cả chính nó, giống cách DBSCAN đếm), sắp xếp tăng dần. Điểm bẻ cong mạnh gợi ý eps.

In [5]:
from sklearn.neighbors import NearestNeighbors

kdist = {}
for ms in MIN_SAMPLES_GRID:
    d = (
        NearestNeighbors(n_neighbors=ms)
        .fit(Z)
        .kneighbors(Z)[0][:, -1]
    )
    kdist[ms] = np.sort(d)

fig, ax = plt.subplots(figsize=(9, 5))
for ms, d in kdist.items():
    ax.plot(d, label=f"min_samples={ms}")

ax.set_ylim(0, 3)
ax.set_xlabel("Khách hàng (sắp xếp theo khoảng cách)")
ax.set_ylabel("Khoảng cách k-distance")
ax.set_title("B03-DB — k-distance (Yeo–Johnson + Standard + PCA 90%)")
ax.legend()

plt.tight_layout()
FIG.mkdir(parents=True, exist_ok=True)
plt.savefig(FIG / "dbscan_b03_k_distance.png", dpi=150, bbox_inches="tight")
plt.show()

display(
    pd.DataFrame(
        {
            ms: np.quantile(d, [0.5, 0.9, 0.95, 0.99])
            for ms, d in kdist.items()
        },
        index=["P50", "P90", "P95", "P99"]
    ).round(3)
)

dbscan.ipynb:cell-11:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


,4,6,8,10,15,20
P50,0.053,0.069,0.082,0.094,0.117,0.136
P90,0.097,0.124,0.145,0.165,0.199,0.234
P95,0.136,0.170,0.203,0.226,0.273,0.316
P99,0.271,0.346,0.408,0.455,0.525,0.592


**Cách đọc:** điểm bẻ cong của đường k-distance gợi ý vùng eps cần xem xét. Lưới eps được giữ giống B01 để đối chiếu; không thay ngưỡng nhiễu sau khi xem điểm số.

## Quét lưới eps × min_samples

Silhouette và DBI chỉ tính trên khách **không bị gán nhiễu** và chỉ khi có ≥ 2 cụm.

In [6]:
def run_dbscan(Zs, eps, ms):
    labels = DBSCAN(eps=eps, min_samples=ms).fit_predict(Zs)
    mask = labels != -1
    sizes = pd.Series(labels[mask]).value_counts()
    k = len(sizes)

    row = {
        "min_samples": ms,
        "eps": eps,
        "n_clusters": k,
        "noise_pct": 100 * (~mask).mean(),
        "largest_pct": 100 * sizes.iloc[0] / len(Zs) if k else np.nan,
        "smallest": int(sizes.iloc[-1]) if k else 0,
        "silhouette": np.nan,
        "dbi": np.nan,
    }

    # Chỉ đánh giá trên các điểm không phải nhiễu
    if 2 <= k < mask.sum():
        row["silhouette"] = silhouette_score(Zs[mask], labels[mask])
        row["dbi"] = davies_bouldin_score(Zs[mask], labels[mask])

    return row, labels


# Z: Yeo–Johnson → StandardScaler → PCA 90% của B03
pairs = [(e, m) for m in MIN_SAMPLES_GRID for e in EPS_GRID]
runs = [run_dbscan(Z, e, m) for e, m in pairs]

grid = pd.DataFrame([row for row, _ in runs])
labels_b03 = {
    pair: labels
    for pair, (_, labels) in zip(pairs, runs)
}

eligible = (
    (grid["n_clusters"] >= MIN_CLUSTERS)
    & (grid["noise_pct"] <= MAX_NOISE * 100)
    & grid["silhouette"].notna()
)

print("Số cấu hình đã chạy:", len(grid))
print(
    "Số cấu hình có >= 2 cụm:",
    int((grid["n_clusters"] >= 2).sum())
)
print(
    "Số cấu hình đạt điều kiện số cụm và nhiễu:",
    int(eligible.sum())
)

display(
    grid.loc[eligible]
    .sort_values("silhouette", ascending=False)
    .head(10)
    .round(4)
)

Số cấu hình đã chạy: 72
Số cấu hình có >= 2 cụm: 23
Số cấu hình đạt điều kiện số cụm và nhiễu: 8


,min_samples,eps,n_clusters,noise_pct,largest_pct,smallest,silhouette,dbi
41,10,0.30,2,1.2018,98.6365,7,0.4412,0.4008
15,6,0.20,2,2.0800,97.7814,6,0.4313,0.3964
40,10,0.25,2,2.0337,97.6427,14,0.2176,0.7194
4,4,0.25,3,0.8320,98.9600,4,0.2160,0.6055
14,6,0.15,3,4.5528,95.1468,6,0.0965,0.6341
3,4,0.20,3,1.4098,98.4285,3,-0.0355,0.6965
27,8,0.20,4,2.8426,96.5796,8,-0.0588,0.5894
2,4,0.15,9,2.7733,96.1867,4,-0.2340,0.5793


**Đánh giá lưới:** xem số cấu hình đạt điều kiện, Silhouette/DBI trên điểm không nhiễu, tỷ lệ nhiễu và kích thước cụm. Các nhận xét số liệu phải dựa trên lần chạy B03 này.

## Chọn cấu hình

Giữ quy tắc B01: ít nhất 2 cụm (không tính nhiễu), nhiễu không quá 5%,
Silhouette cao nhất trong các cấu hình đạt điều kiện và có điểm hợp lệ.
Nếu không có cấu hình đạt, ghi nhận rõ và vẫn lưu lưới cùng lambda đã fit.

In [7]:
valid = grid.loc[eligible].sort_values('silhouette', ascending=False)
best = None
selected_model = None
labels = None
if valid.empty:
    print('Không có cấu hình B03 đạt điều kiện; không tự nới ngưỡng để chọn model.')
else:
    best = valid.iloc[0]
    print('Cấu hình được chọn:')
    print(best.to_string())
    labels = labels_b03[(float(best['eps']), int(best['min_samples']))]
    selected_model = DBSCAN(eps=float(best['eps']), min_samples=int(best['min_samples'])).fit(Z)
    assert np.array_equal(selected_model.labels_, labels)

Cấu hình được chọn:
min_samples    10.000000
eps             0.300000
n_clusters      2.000000
noise_pct       1.201756
largest_pct    98.636469
smallest        7.000000
silhouette      0.441224
dbi             0.400832


## Hồ sơ cụm

Nhãn `-1` là nhiễu. Cột `Monetary_share_pct` là tỷ trọng tổng Monetary của nhóm.

In [8]:
prof = None
if labels is None:
    print('Chưa có model đạt điều kiện để mô tả cụm.')
else:
    prof = X.assign(cluster=labels).groupby('cluster').agg(
        customers=('Monetary', 'size'),
        Recency=('Recency', 'median'),
        Frequency=('Frequency', 'median'),
        Monetary=('Monetary', 'median'),
        Monetary_sum=('Monetary', 'sum'))
    prof['customer_share_pct'] = 100 * prof['customers'] / len(X)
    total_monetary = prof['Monetary_sum'].sum()
    prof['Monetary_share_pct'] = 100 * prof['Monetary_sum'] / total_monetary if total_monetary > 0 else np.nan
    display(prof.drop(columns='Monetary_sum').round(2))

,customers,Recency,Frequency,Monetary,customer_share_pct,Monetary_share_pct
cluster,,,,,,
-1,52,60.5,2.0,855.43,1.20,19.08
0,4268,51.0,2.0,657.26,98.64,75.83
1,7,3.0,50.0,58762.08,0.16,5.09


**Hồ sơ cụm:** dùng RFM gốc để diễn giải, không dùng giá trị sau power transform. Nhãn -1 là nhiễu theo mật độ, không tự đồng nghĩa khách có giá trị thấp.

In [9]:
if labels is None:
    print('Bỏ qua hình cụm: không có cấu hình đạt điều kiện.')
else:
    fig, ax = plt.subplots(figsize=(9, 6))
    noise = labels == -1
    y = Z[:, 1] if Z.shape[1] >= 2 else np.zeros(len(Z))
    ax.scatter(Z[~noise, 0], y[~noise], c=labels[~noise], cmap='tab10', s=14, alpha=0.6, label='Cụm')
    ax.scatter(Z[noise, 0], y[noise], c='black', marker='x', s=30, label='Nhiễu')
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2' if Z.shape[1] >= 2 else 'Trục phụ bằng 0 (chỉ giữ 1 PC)')
    ax.set_title(f"B03-DB — eps={best['eps']}, min_samples={int(best['min_samples'])} (Yeo–Johnson; PCA 90%)")
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIG / 'dbscan_b03_clusters_pca.png', dpi=150)
    plt.show()

dbscan.ipynb:cell-21:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


**Trực quan hóa:** hình dùng tối đa hai thành phần đầu. Kết hợp hình với tỷ lệ nhiễu, kích thước và hồ sơ cụm trước khi kết luận chất lượng B03.

## Lưu kết quả B03 và lambda đã fit


In [10]:
grid.to_csv(OUT / 'dbscan_b03_grid.csv', index=False)
if labels is not None:
    pd.DataFrame({'CustomerID': df['CustomerID'], 'cluster': labels}).to_csv(
        OUT / 'dbscan_b03_labels.csv', index=False)

preprocessor = Pipeline([('power', power), ('scaler', scaler), ('pca', pca_b03)])
params = {
    'experiment': EXPERIMENT, 'parent_experiment': 'B01-DB', 'features': FEATURES,
    'data_sha256': sha256(DATA.read_bytes()).hexdigest(),
    'transform': 'Yeo-Johnson', 'method': POWER_METHOD, 'standardize': POWER_STANDARDIZE,
    'lambda_by_feature': lambda_by_feature, 'scaler': 'StandardScaler',
    'pca_variance': PCA_VAR, 'n_components': n_pc,
    'pca_variance_kept': float(pca_b03.explained_variance_ratio_.sum()),
    'eps_grid': EPS_GRID, 'min_samples_grid': MIN_SAMPLES_GRID,
    'max_noise': MAX_NOISE, 'min_clusters': MIN_CLUSTERS,
    'selection_status': 'selected' if best is not None else 'no_eligible_configuration',
    'selection_rule': 'max silhouette_non_noise subject to min_clusters and max_noise',
    'versions': {'numpy': np.__version__, 'pandas': pd.__version__, 'sklearn': sklearn.__version__},
}
if best is not None:
    params.update({
        'eps': float(best['eps']), 'min_samples': int(best['min_samples']),
        'n_clusters': int(best['n_clusters']), 'noise_pct': float(best['noise_pct']),
        'silhouette_non_noise': float(best['silhouette']), 'dbi_non_noise': float(best['dbi']),
    })
artifact = {
    'features': FEATURES, 'preprocessor': preprocessor, 'dbscan': selected_model,
    'customer_ids': df['CustomerID'].tolist(), 'params': params,
}
artifact_path = BASE_DIR / 'dbscan_b03_artifacts.joblib'
joblib.dump(artifact, artifact_path)
loaded = joblib.load(artifact_path)
assert np.allclose(loaded['preprocessor'].transform(X), Z)
assert np.array_equal(loaded['preprocessor']['power'].lambdas_, power.lambdas_)
if labels is not None:
    assert np.array_equal(loaded['dbscan'].labels_, labels)
with (OUT / 'dbscan_b03_params.json').open('w', encoding='utf-8') as f:
    json.dump(params, f, ensure_ascii=False, indent=2, allow_nan=False)
print('Đã lưu tham số, lambda và artifact B03; kiểm tra nạp lại khớp.')
print('Trạng thái:', params['selection_status'])

Đã lưu tham số, lambda và artifact B03; kiểm tra nạp lại khớp.
Trạng thái: selected
